In [1]:
# =============================================================================
# SatMAE (ViT-S/16) — Masked Image Modeling for Satellite Imagery
# Pre-train on PatternNet → Evaluate on EuroSAT-RGB / EuroSAT-MS
#
# ── Comparability changes vs original SatMAE ─────────────────────────────────
#   C1.  arch: vit_base_patch16_224 → vit_small_patch16_224  (matches DINO)
#   C2.  img_size: 224 → 160                                  (matches DINO S20)
#   C3.  embed_dim: 768 → 384                                 (matches DINO ViT-S)
#   C4.  decoder_dim: 512 → 256, depth 8→4, heads 16→8       (scaled to ViT-S)
#   C5.  batch_size: 64 → 512                                 (matches DINO)
#   C6.  lr: 1.5e-4 → 1.2e-3  (linear scale: 1.5e-4×512/64) (matches DINO)
#   C7.  ensure_split() for consistent 80/20 eval split       (matches DINO)
#   C8.  eval crop: 224 → 160  (Resize 182 → CenterCrop 160) (matches DINO)
#   C9.  timm.create_model passes img_size=160                (matches DINO)
#
# ── DINO-parity fixes ─────────────────────────────────────────────────────────
#   P1.  Albumentations augmentation pipeline (falls back to torchvision).
#   P2.  channels_last removed from load_backbone and all eval hot-paths.
#   P3.  get_eval_transforms() honours img_size=160 everywhere.
#   P4.  TwoViewDataset supports Albumentations.
#   P5.  batched_knn_predict() matching DINO S6 scatter_add, chunked.
#   P6.  async_save() + threading for non-blocking checkpoint I/O.
#   P7.  torch.compile applied to full model AFTER DataParallel (BUG FIX).
#   P8.  BandAdapterBackbone (trainable 1×1 conv) for band-mismatch eval.
#   P9.  MultiSpectralDataset uses CFG["img_size"] (not hardcoded 224).
#   P10. SegDecoder uses CFG["img_size"] (not hardcoded 224).
#   P11. _make_backbone() helper shared across all construction paths.
#
# ── Key Bug Fix (P7) ──────────────────────────────────────────────────────────
#   nn.DataParallel.replicate() calls deepcopy on every forward pass.
#   deepcopy of a torch._dynamo.OptimizedModule does NOT rebind submodule
#   attributes (e.g. patch_embed) through __getattr__ on the replica →
#   AttributeError on device replicas.
#
#   Wrong order (original):  compile(encoder) → DataParallel(model)
#   Correct order (fixed):   DataParallel(model) → compile(model)
#
#   Compiling the outer model after DP means deepcopy never sees a compiled
#   submodule; torch.compile traces through DP's forward cleanly.
# =============================================================================

# !pip install timm torchvision torch einops scipy scikit-learn albumentations --quiet

import os
import math
import random
import json
import time
import threading
import shutil
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, datasets
import timm
from einops import rearrange
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedShuffleSplit

# ── Albumentations (optional) ─────────────────────────────────────────────────
try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    HAS_ALBU = True
except ImportError:
    HAS_ALBU = False
    print("albumentations not found — falling back to torchvision transforms.")

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark     = True
torch.set_float32_matmul_precision("high")

# ── Environment ───────────────────────────────────────────────────────────────
BASE_DIR    = "/kaggle/working"
DATA_DIR    = "/kaggle/input"
NUM_WORKERS = 4
COMPILE     = torch.__version__ >= "2.0.0"

_PATTERNNET_DIR  = (
    f"{DATA_DIR}/datasets/samitsaleem/"
    "patternnet-scene-classification-dataset/PatternNet_Images"
)
_EUROSAT_RGB_DIR = f"{DATA_DIR}/datasets/pranjallk1995/rgbeurosat/RBG"
_EUROSAT_MS_DIR  = (
    f"{DATA_DIR}/datasets/nguyenquangnhat2100/eurosatallbands"
    "/ds/images/remote_sensing/otherDatasets/sentinel_2/tif"
)

# ── AMP dtype ─────────────────────────────────────────────────────────────────
def _amp_dtype():
    if not torch.cuda.is_available():
        return None
    return (torch.bfloat16
            if torch.cuda.get_device_capability()[0] >= 8
            else torch.float16)

AMP_DTYPE = _amp_dtype()

# ── Config ──────────────────��─────────────────────────────────────────────────
CFG = dict(
    patternnet_dir  = _PATTERNNET_DIR,
    eurosat_rgb_dir = _EUROSAT_RGB_DIR,
    eurosat_ms_dir  = _EUROSAT_MS_DIR,
    output_dir      = f"{BASE_DIR}/satmae",
    checkpoint      = None,

    # Architecture (scaled to match DINO ViT-S)
    arch           = "vit_small_patch16_224",
    img_size       = 160,
    patch_size     = 16,
    in_channels    = 3,
    embed_dim      = 384,
    decoder_dim    = 256,
    decoder_depth  = 4,
    decoder_heads  = 8,
    mask_ratio     = 0.75,

    # Training (matched to DINO regime)
    epochs         = 200,
    batch_size     = 512,   # reduce to 256 on a single T4
    lr             = 1.2e-3,
    min_lr         = 1e-6,
    weight_decay   = 0.05,
    warmup_epochs  = 20,
    norm_pix_loss  = True,

    # Augmentation
    crop_scale      = (0.2, 1.0),
    jitter_strength = 0.4,
    blur_prob       = 0.5,

    # Evaluation
    num_classes       = 10,
    knn_k             = 20,
    retrieval_ks      = [1, 5, 10],
    geo_thresholds_km = [1, 5, 10],
    val_split         = 0.2,
)

os.makedirs(CFG["output_dir"], exist_ok=True)


# =============================================================================
# ── Utilities ─────────────────────────────────────────────────────────────────
# =============================================================================

def ensure_split(root, val_frac=0.2, seed=SEED):
    """Deterministic 80/20 stratified train/val split (hard-links or copies)."""
    if os.path.isdir(os.path.join(root, "train")):
        return root
    split_root = root.rstrip("/") + "_split"
    if os.path.isdir(os.path.join(split_root, "train")):
        print(f"  Using cached split at {split_root}")
        return split_root
    print(f"  Creating 80/20 stratified split → {split_root} …")
    base   = datasets.ImageFolder(root)
    labels = np.array([y for _, y in base.samples])
    sss    = StratifiedShuffleSplit(1, test_size=val_frac, random_state=seed)
    train_idx, val_idx = next(sss.split(np.zeros(len(labels)), labels))
    for split_name, indices in [("train", train_idx), ("val", val_idx)]:
        for idx in indices:
            src, cls_idx = base.samples[idx]
            cls_name     = base.classes[cls_idx]
            dst_dir      = os.path.join(split_root, split_name, cls_name)
            os.makedirs(dst_dir, exist_ok=True)
            dst = os.path.join(dst_dir, os.path.basename(src))
            if not os.path.exists(dst):
                try:
                    os.link(src, dst)
                except OSError:
                    shutil.copy2(src, dst)
    print(f"  Split: {len(train_idx)} train / {len(val_idx)} val")
    return split_root


def make_loader(ds, batch_size, shuffle, drop_last=False,
                collate_fn=None, sampler=None):
    pw = NUM_WORKERS > 0
    return DataLoader(
        ds,
        batch_size         = batch_size,
        shuffle            = (shuffle if sampler is None else False),
        sampler            = sampler,
        num_workers        = NUM_WORKERS,
        pin_memory         = True,
        drop_last          = drop_last,
        persistent_workers = pw,
        prefetch_factor    = 4 if pw else None,
        collate_fn         = collate_fn,
    )


_save_thread: threading.Thread = None

def async_save(path, obj):
    """Non-blocking checkpoint save (P6)."""
    global _save_thread
    if _save_thread is not None:
        _save_thread.join()

    def _save():
        torch.save(obj, path)
        print(f"  → Saved {path}", flush=True)

    _save_thread = threading.Thread(target=_save, daemon=True)
    _save_thread.start()


# =============================================================================
# ── Augmentations ─────────────────────────────────────────────────────────────
# =============================================================================

def _base_aug_albu(size, scale, jitter_s, blur_prob, solarize_prob=0.0):
    ops = [
        A.RandomResizedCrop(size=(size, size), scale=scale, interpolation=3),
        A.HorizontalFlip(p=0.5),
        A.ColorJitter(
            brightness=jitter_s, contrast=jitter_s,
            saturation=jitter_s, hue=jitter_s * 0.25, p=0.8,
        ),
        A.ToGray(p=0.2),
        A.GaussianBlur(blur_limit=(9, 9), sigma_limit=(0.1, 2.0), p=blur_prob),
    ]
    if solarize_prob > 0:
        ops.append(A.Solarize(threshold=128, p=solarize_prob))
    ops += [
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ]
    return A.Compose(ops)


def _base_aug_tv(size, scale, jitter_s, blur_prob, solarize_prob=0.0):
    ops = [
        transforms.RandomResizedCrop(size, scale=scale, interpolation=3),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomApply(
            [transforms.ColorJitter(
                brightness=jitter_s, contrast=jitter_s,
                saturation=jitter_s, hue=jitter_s * 0.25)],
            p=0.8,
        ),
        transforms.RandomGrayscale(p=0.2),
        transforms.RandomApply(
            [transforms.GaussianBlur(kernel_size=9, sigma=(0.1, 2.0))],
            p=blur_prob,
        ),
    ]
    if solarize_prob > 0:
        ops.append(transforms.RandomSolarize(threshold=128, p=solarize_prob))
    ops += [
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
    return transforms.Compose(ops)


def _base_aug(size, scale, jitter_s, blur_prob, solarize_prob=0.0):
    return (
        _base_aug_albu(size, scale, jitter_s, blur_prob, solarize_prob)
        if HAS_ALBU else
        _base_aug_tv(size, scale, jitter_s, blur_prob, solarize_prob)
    )


def get_pretrain_aug(img_size, crop_scale):
    """Minimal augmentation for MAE pre-training (no colour jitter)."""
    if HAS_ALBU:
        return A.Compose([
            A.RandomResizedCrop(
                size=(img_size, img_size), scale=crop_scale, interpolation=3),
            A.HorizontalFlip(p=0.5),
            A.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])
    return transforms.Compose([
        transforms.RandomResizedCrop(img_size, scale=crop_scale,
                                     interpolation=3),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])


def get_eval_transforms():
    """C8 / P3: eval crops at CFG img_size (160), not hardcoded 224."""
    img_size = CFG["img_size"]
    resize   = int(img_size * (256 / 224) + 0.5)   # 182 @ 160 px
    val_tf = transforms.Compose([
        transforms.Resize(resize),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(img_size),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    return train_tf, val_tf


# =============================================================================
# ── Datasets ──────────────────────────────────────────────────────────────────
# =============================================================================

class PatternNetDataset(Dataset):
    def __init__(self, root, aug):
        self.base     = datasets.ImageFolder(root)
        self.aug      = aug
        self.use_albu = HAS_ALBU

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, _ = self.base[idx]
        if self.use_albu:
            return self.aug(image=np.array(img))["image"]
        return self.aug(img)


class TwoViewDataset(Dataset):
    """Returns two independently augmented views of the same image (P4)."""
    def __init__(self, root, aug):
        self.base     = datasets.ImageFolder(root)
        self.aug      = aug
        self.use_albu = HAS_ALBU

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, label = self.base[idx]
        if self.use_albu:
            arr = np.array(img)
            x1  = self.aug(image=arr)["image"]
            x2  = self.aug(image=arr)["image"]
        else:
            x1 = self.aug(img)
            x2 = self.aug(img)
        return x1, x2, label


class MultiSpectralDataset(Dataset):
    """Supports .tif (rasterio), .npy, .png/.jpg.  P9: uses CFG img_size."""
    def __init__(self, root, n_channels=13, img_size=None):
        self.samples    = []
        self.n_channels = n_channels
        self.img_size   = img_size or CFG["img_size"]
        classes = sorted(
            d for d in os.listdir(root)
            if os.path.isdir(os.path.join(root, d))
        )
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        for cls in classes:
            cls_dir = os.path.join(root, cls)
            for fname in sorted(os.listdir(cls_dir)):
                if fname.lower().endswith(
                        (".tif", ".tiff", ".npy", ".png", ".jpg")):
                    self.samples.append(
                        (os.path.join(cls_dir, fname),
                         self.class_to_idx[cls])
                    )

    def __len__(self):
        return len(self.samples)

    def _load_tif(self, path):
        try:
            import rasterio
        except ImportError:
            raise ImportError("pip install rasterio")
        with rasterio.open(path) as src:
            arr = src.read().astype(np.float32)
        n = self.n_channels
        if arr.shape[0] >= n:
            arr = arr[:n]
        else:
            pad = np.zeros(
                (n - arr.shape[0], *arr.shape[1:]), dtype=np.float32)
            arr = np.concatenate([arr, pad], axis=0)
        return torch.from_numpy(arr)

    def _normalise(self, x):
        mean = x.view(x.shape[0], -1).mean(1, keepdim=True).unsqueeze(-1)
        std  = (
            x.view(x.shape[0], -1).std(1, keepdim=True)
             .unsqueeze(-1).clamp(min=1e-6)
        )
        return (x - mean) / std

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        ext = os.path.splitext(path)[1].lower()

        if ext in (".tif", ".tiff"):
            x = self._load_tif(path)

        elif ext == ".npy":
            arr = np.load(path).astype(np.float32)
            if arr.ndim == 3 and arr.shape[2] == self.n_channels:
                arr = arr.transpose(2, 0, 1)
            x = torch.from_numpy(arr)
            if x.shape[0] > self.n_channels:
                x = x[:self.n_channels]

        else:
            from PIL import Image
            img    = Image.open(path).convert("RGB")
            resize = int(self.img_size * (256 / 224) + 0.5)
            x = transforms.Compose([
                transforms.Resize(resize),
                transforms.CenterCrop(self.img_size),
                transforms.ToTensor(),
                transforms.Normalize(
                    [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
            ])(img)
            # Repeat RGB channels to fill n_channels slots
            x = x.repeat(math.ceil(self.n_channels / 3), 1, 1)[:self.n_channels]
            return x, label

        x = self._normalise(x)
        x = F.interpolate(
            x[None], size=self.img_size,
            mode="bilinear", align_corners=False,
        )[0]
        return x, label


# =============================================================================
# ── Backbone factory (P11) ────────────────────────────────────────────────────
# =============================================================================

def _make_backbone(arch, img_size=160):
    """Shared timm ViT constructor — identical path for train and eval."""
    m = timm.create_model(
        arch,
        pretrained=False,
        num_classes=0,
        img_size=img_size,
        dynamic_img_size=True,
    )
    if hasattr(m, "set_attn_backend"):
        try:
            m.set_attn_backend("sdpa")
        except Exception:
            pass
    return m


# =============================================================================
# ── SatMAE Encoder (ViT-S/16 @ 160 px) ───────────────────────────────────────
# =============================================================================

class SatMAEEncoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.img_size   = cfg["img_size"]
        self.patch_size = cfg["patch_size"]
        self.in_ch      = cfg["in_channels"]
        self.embed_dim  = cfg["embed_dim"]
        n_patches = (cfg["img_size"] // cfg["patch_size"]) ** 2   # 100 @ 160 px

        # Own patch projection (not borrowed from timm)
        self.patch_embed = nn.Conv2d(
            self.in_ch, self.embed_dim,
            kernel_size=cfg["patch_size"],
            stride=cfg["patch_size"],
        )
        self.cls_token = nn.Parameter(torch.zeros(1, 1, self.embed_dim))
        self.pos_embed = nn.Parameter(
            torch.zeros(1, n_patches + 1, self.embed_dim),
            requires_grad=False,
        )

        # Borrow transformer blocks and final norm from timm ViT-S (P11)
        vit         = _make_backbone(cfg["arch"], cfg["img_size"])
        self.blocks = vit.blocks
        self.norm   = vit.norm

        self._init_pos_embed(n_patches)
        self._init_weights()

    # ── Sinusoidal positional embedding ──────────────────────────────────────
    def _init_pos_embed(self, n_patches):
        pos      = torch.zeros(n_patches + 1, self.embed_dim)
        position = torch.arange(
            0, n_patches + 1, dtype=torch.float).unsqueeze(1)
        div = torch.exp(
            torch.arange(0, self.embed_dim, 2).float() *
            (-math.log(10_000.0) / self.embed_dim)
        )
        pos[1:, 0::2] = torch.sin(position[1:] * div)
        pos[1:, 1::2] = torch.cos(position[1:] * div)
        self.pos_embed.data.copy_(pos.unsqueeze(0))

    def _init_weights(self):
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.xavier_uniform_(
            self.patch_embed.weight.view(
                self.patch_embed.weight.size(0), -1))

    # ── Random masking ────────────────────────────────────────────────────────
    def random_masking(self, x, mask_ratio):
        B, N, D = x.shape
        n_keep  = int(N * (1 - mask_ratio))
        noise       = torch.rand(B, N, device=x.device)
        ids_shuffle = noise.argsort(dim=1)
        ids_restore = ids_shuffle.argsort(dim=1)
        ids_keep    = ids_shuffle[:, :n_keep]
        x_masked    = x.gather(
            1, ids_keep.unsqueeze(-1).expand(-1, -1, D))
        mask        = torch.ones(B, N, device=x.device)
        mask.scatter_(1, ids_keep, 0)
        return x_masked, mask, ids_restore

    # ── Forward ───────────────────────────────────────────────────────────────
    def forward(self, x, mask_ratio=0.75):
        x = self.patch_embed(x)                          # (B, D, H', W')
        x = rearrange(x, "b d h w -> b (h w) d")         # (B, N, D)
        x = x + self.pos_embed[:, 1:]
        x, mask, ids_restore = self.random_masking(x, mask_ratio)
        cls = (self.cls_token + self.pos_embed[:, :1]).expand(
            x.shape[0], -1, -1)
        x   = torch.cat([cls, x], dim=1)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        return x, mask, ids_restore

    def forward_features(self, x):
        """All tokens visible (mask_ratio=0) — used during eval."""
        latent, _, _ = self.forward(x, mask_ratio=0.0)
        return latent   # (B, N+1, D)


# =============================================================================
# ── SatMAE Decoder (scaled to ViT-S, C4) ─────────────────────────────────────
# =============================================================================

class SatMAEDecoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        n_patches = (cfg["img_size"] // cfg["patch_size"]) ** 2
        patch_dim = cfg["in_channels"] * cfg["patch_size"] ** 2   # 768

        self.embed      = nn.Linear(cfg["embed_dim"], cfg["decoder_dim"])
        self.mask_token = nn.Parameter(
            torch.zeros(1, 1, cfg["decoder_dim"]))
        self.pos_embed  = nn.Parameter(
            torch.zeros(1, n_patches + 1, cfg["decoder_dim"]),
            requires_grad=False,
        )

        layer = nn.TransformerEncoderLayer(
            d_model         = cfg["decoder_dim"],
            nhead           = cfg["decoder_heads"],
            dim_feedforward = cfg["decoder_dim"] * 4,
            batch_first     = True,
            norm_first      = True,
        )
        self.blocks = nn.TransformerEncoder(
            layer, num_layers=cfg["decoder_depth"])
        self.norm   = nn.LayerNorm(cfg["decoder_dim"])
        self.head   = nn.Linear(cfg["decoder_dim"], patch_dim)

        self._init_pos_embed(n_patches, cfg["decoder_dim"])
        nn.init.trunc_normal_(self.mask_token, std=0.02)

    def _init_pos_embed(self, n_patches, dim):
        pos      = torch.zeros(n_patches + 1, dim)
        position = torch.arange(0, n_patches + 1).float().unsqueeze(1)
        div = torch.exp(
            torch.arange(0, dim, 2).float() *
            (-math.log(10_000.0) / dim)
        )
        pos[1:, 0::2] = torch.sin(position[1:] * div)
        pos[1:, 1::2] = torch.cos(position[1:] * div)
        self.pos_embed.data.copy_(pos.unsqueeze(0))

    def forward(self, x, ids_restore):
        x       = self.embed(x)
        B, _, D = x.shape
        N       = ids_restore.shape[1]
        n_mask  = N - (x.shape[1] - 1)

        mask_tokens = self.mask_token.expand(B, n_mask, -1)
        x_no_cls    = torch.cat([x[:, 1:], mask_tokens], dim=1)
        x_no_cls    = x_no_cls.gather(
            1, ids_restore.unsqueeze(-1).expand(-1, -1, D))
        x = torch.cat([x[:, :1], x_no_cls], dim=1)
        x = x + self.pos_embed
        x = self.blocks(x)
        x = self.norm(x)
        x = self.head(x[:, 1:])
        return x


# =============================================================================
# ── Full SatMAE model ─────────────────────────────────────────────────────────
# =============================================================================

class SatMAE(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.encoder    = SatMAEEncoder(cfg)
        self.decoder    = SatMAEDecoder(cfg)
        self.patch_size = cfg["patch_size"]
        self.in_ch      = cfg["in_channels"]
        self.norm_pix   = cfg["norm_pix_loss"]

    def patchify(self, x):
        p = self.patch_size
        return rearrange(
            x, "b c (h p1) (w p2) -> b (h w) (p1 p2 c)", p1=p, p2=p)

    def forward(self, x, mask_ratio=0.75):
        latent, mask, ids_restore = self.encoder(x, mask_ratio)
        pred   = self.decoder(latent, ids_restore)
        target = self.patchify(x)
        if self.norm_pix:
            mean   = target.mean(dim=-1, keepdim=True)
            var    = target.var(dim=-1,  keepdim=True)
            target = (target - mean) / (var + 1e-6).sqrt()
        loss = ((pred - target) ** 2).mean(dim=-1)
        loss = (loss * mask).sum() / mask.sum()
        return loss


# =============================================================================
# ── LR schedule — cosine with linear warmup ───────────────────────────────────
# =============================================================================

def adjust_lr(optimizer, epoch, cfg):
    if epoch < cfg["warmup_epochs"]:
        lr = cfg["lr"] * (epoch + 1) / cfg["warmup_epochs"]
    else:
        progress = (epoch - cfg["warmup_epochs"]) / max(
            1, cfg["epochs"] - cfg["warmup_epochs"])
        lr = cfg["min_lr"] + 0.5 * (cfg["lr"] - cfg["min_lr"]) * (
            1 + math.cos(math.pi * progress))
    for g in optimizer.param_groups:
        g["lr"] = lr
    return lr


# =============================================================================
# ── Training ──────────────────────────────────────────────────────────────────
# =============================================================================

def train():
    device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp   = AMP_DTYPE is not None
    use_scaler = AMP_DTYPE == torch.float16

    print(
        f"Device: {device}  |  Workers: {NUM_WORKERS}  |  "
        f"AMP: {AMP_DTYPE}  |  compile: {COMPILE}  |  "
        f"albu: {HAS_ALBU}  |  batch: {CFG['batch_size']}"
    )

    aug     = get_pretrain_aug(CFG["img_size"], CFG["crop_scale"])
    dataset = PatternNetDataset(CFG["patternnet_dir"], aug)
    loader  = make_loader(
        dataset, CFG["batch_size"], shuffle=True, drop_last=True)

    # ── Build model ───────────────────────────────────────────────────────────
    model = SatMAE(CFG).to(device)
    # channels_last benefits the Conv2d patch_embed during pre-training only;
    # all eval paths use plain contiguous tensors (P2).
    model = model.to(memory_format=torch.channels_last)

    # ── DataParallel FIRST, then torch.compile (P7 BUG FIX) ──────────────────
    #
    # Root cause of the original crash:
    #   nn.DataParallel.replicate() calls deepcopy() on every forward pass.
    #   deepcopy of a torch._dynamo.OptimizedModule does NOT rebind submodule
    #   attributes (patch_embed, cls_token, …) through __getattr__ on the
    #   replica object → AttributeError on device 0 and all other replicas.
    #
    # Wrong (original): compile(encoder) → DataParallel(model)
    #   deepcopy sees a compiled encoder → replica __getattr__ breaks.
    #
    # Correct (this code): DataParallel(model) → compile(outer model)
    #   deepcopy happens on plain nn.Module sub-objects before compile.
    #   torch.compile then traces through DataParallel's forward cleanly;
    #   no subsequent deepcopy ever touches a compiled object.
    #
    multi_gpu = torch.cuda.device_count() > 1
    if multi_gpu:
        model = nn.DataParallel(model)
        print(f"DataParallel on {torch.cuda.device_count()} GPUs.")

    if COMPILE:
        try:
            model = torch.compile(model, mode="reduce-overhead")
            print("torch.compile enabled on full SatMAE model.")
        except Exception as e:
            print(f"torch.compile skipped: {e}")

    # ── Optimiser ─────────────────────────────────────────────────────────────
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr           = CFG["lr"],
        weight_decay = CFG["weight_decay"],
        fused        = True,
    )
    scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)

    start_epoch = 0
    if CFG["checkpoint"]:
        ckpt = torch.load(CFG["checkpoint"], map_location="cpu")
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        start_epoch = ckpt["epoch"] + 1
        print(f"Resumed from epoch {start_epoch}")

    # ── Training loop ─────────────────────────────────────────────────────────
    log           = []
    t_train_start = time.time()

    for epoch in range(start_epoch, CFG["epochs"]):
        model.train()
        total_loss = 0.0
        t0         = time.time()
        lr         = adjust_lr(optimizer, epoch, CFG)

        for x in loader:
            x = x.to(device, non_blocking=True,
                     memory_format=torch.channels_last)
            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(
                    "cuda", dtype=AMP_DTYPE, enabled=use_amp):
                loss = model(x, CFG["mask_ratio"])
                loss = loss.mean()   # safe for DataParallel

            if torch.isnan(loss) or torch.isinf(loss):
                print(
                    f"  NaN/Inf loss at epoch {epoch+1} — skipping batch.")
                optimizer.zero_grad(set_to_none=True)
                continue

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=3.0)
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()

        avg_loss = total_loss / max(1, len(loader))
        elapsed  = time.time() - t0
        log.append({
            "epoch":        epoch,
            "loss":         avg_loss,
            "lr":           lr,
            "epoch_time_s": round(elapsed, 1),
        })
        print(
            f"Epoch [{epoch+1:>3}/{CFG['epochs']}]  "
            f"loss={avg_loss:.4f}  lr={lr:.2e}  time={elapsed:.0f}s"
        )

        if (epoch + 1) % 50 == 0 or epoch == CFG["epochs"] - 1:
            path = os.path.join(
                CFG["output_dir"], f"satmae_ep{epoch+1}.pt")
            # Unwrap compile and/or DataParallel to get raw state_dict
            raw = model
            if hasattr(raw, "_orig_mod"):       # compiled
                raw = raw._orig_mod
            if hasattr(raw, "module"):          # DataParallel
                raw = raw.module
            async_save(path, {
                "epoch":     epoch,
                "model":     raw.state_dict(),
                "optimizer": optimizer.state_dict(),
                "cfg":       CFG,
            })

    if _save_thread is not None:
        _save_thread.join()

    with open(
            os.path.join(CFG["output_dir"], "satmae_log.json"), "w") as f:
        json.dump(log, f, indent=2)

    total = time.time() - t_train_start
    print(
        f"\nPre-training done.  Total: {total/3600:.2f} h ({total:.0f} s)")


# =============================================================================
# ── Backbone loading (eval) ───────────────────────────────────────────────────
# =============================================================================

def load_backbone(backbone_path, device):
    """Load frozen SatMAE encoder.  P2: no channels_last for ViT eval."""
    ckpt  = torch.load(backbone_path, map_location="cpu")
    state = {
        k.replace("module.", "").replace("encoder.", ""): v
        for k, v in ckpt["model"].items()
        if "encoder." in k
    }
    encoder = SatMAEEncoder(CFG)
    encoder.load_state_dict(state, strict=False)
    encoder.eval().to(device)   # plain .to() — no channels_last (P2)
    for p in encoder.parameters():
        p.requires_grad_(False)
    return encoder


# =============================================================================
# ── Feature extraction helpers ────────────────────────────────────────────────
# =============================================================================

def extract_feat(encoder, x):
    """CLS token, all patches visible (mask_ratio=0).  P2: no channels_last."""
    with torch.no_grad():
        latent, _, _ = encoder(x, mask_ratio=0.0)
        return latent[:, 0]   # (B, embed_dim)


@torch.inference_mode()
def extract_features(encoder, loader, device):
    all_feats, all_labels = [], []
    use_amp = AMP_DTYPE is not None
    encoder.eval()
    for x, y in loader:
        with torch.amp.autocast(
                "cuda", dtype=AMP_DTYPE, enabled=use_amp):
            feats = extract_feat(
                encoder, x.to(device, non_blocking=True))
        all_feats.append(feats.float().cpu())
        all_labels.append(y)
    return torch.cat(all_feats), torch.cat(all_labels)


# =============================================================================
# ── P5: batched kNN predict (scatter_add, matches DINO S6) ───────────────────
# =============================================================================

def batched_knn_predict(sim, train_labels, k, num_classes, chunk=512):
    N_val  = sim.shape[0]
    device = sim.device
    preds  = torch.empty(N_val, dtype=torch.long, device=device)
    for start in range(0, N_val, chunk):
        end        = min(start + chunk, N_val)
        top_idx    = sim[start:end].topk(k, dim=1).indices
        top_labels = train_labels.to(device)[top_idx]
        B          = end - start
        votes      = torch.zeros(B, num_classes, device=device)
        votes.scatter_add_(
            1, top_labels, torch.ones(B, k, device=device))
        preds[start:end] = votes.argmax(1)
    return preds


# =============================================================================
# §4.1  CLASSIFICATION — linear probe
# =============================================================================

def eval_classification(backbone_path, eurosat_dir,
                        label_fracs=(0.01, 0.1, 1.0), num_classes=10):
    device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    encoder = load_backbone(backbone_path, device)
    train_tf, val_tf = get_eval_transforms()

    split_dir  = ensure_split(eurosat_dir, CFG["val_split"])
    train_full = datasets.ImageFolder(
        os.path.join(split_dir, "train"), transform=train_tf)
    val_ds     = datasets.ImageFolder(
        os.path.join(split_dir, "val"),   transform=val_tf)

    print("  Pre-extracting features for linear probe…")
    all_train_feats, all_train_labels = extract_features(
        encoder, make_loader(train_full, 512, shuffle=False), device)
    val_feats, val_labels = extract_features(
        encoder, make_loader(val_ds, 512, shuffle=False), device)

    all_train_feats  = all_train_feats.to(device)
    all_train_labels = all_train_labels.to(device)
    val_feats_dev    = val_feats.to(device)
    val_labels_dev   = val_labels.to(device)

    results = {}
    bs      = 256

    for frac in label_fracs:
        n       = max(num_classes, int(len(train_full) * frac))
        indices = random.sample(range(len(all_train_feats)), n)
        idx_t   = torch.tensor(indices, device=device)
        f_sub   = all_train_feats[idx_t]
        l_sub   = all_train_labels[idx_t]

        head  = nn.Linear(CFG["embed_dim"], num_classes).to(device)
        opt   = torch.optim.SGD(
            head.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(
            opt, T_max=100)

        for _ in range(100):
            head.train()
            perm = torch.randperm(len(f_sub), device=device)
            for start in range(0, len(f_sub), bs):
                sel = perm[start:start + bs]
                opt.zero_grad()
                F.cross_entropy(head(f_sub[sel]), l_sub[sel]).backward()
                opt.step()
            sched.step()

        head.eval()
        with torch.no_grad():
            preds = torch.cat([
                head(val_feats_dev[s:s + bs]).argmax(1)
                for s in range(0, len(val_feats_dev), bs)
            ]).cpu().numpy()
        labels_np = val_labels_dev.cpu().numpy()

        acc      = 100.0 * (preds == labels_np).mean()
        macro_f1 = 100.0 * f1_score(labels_np, preds, average="macro")
        key = f"{int(round(frac * 100))}pct"
        results[key] = {
            "top1_acc": round(acc, 2),
            "macro_f1": round(macro_f1, 2),
        }
        print(
            f"  [{int(frac*100)}% labels]  "
            f"Top-1={acc:.2f}%  Macro-F1={macro_f1:.2f}%"
        )

    return results


# =============================================================================
# §4.2  SEGMENTATION — frozen encoder + linear pixel decoder
# =============================================================================

class SegDecoder(nn.Module):
    """P10: uses CFG img_size, not hardcoded 224."""
    def __init__(self, embed_dim, num_classes,
                 patch_size=16, img_size=160):
        super().__init__()
        self.grid_size = img_size // patch_size
        self.img_size  = img_size
        self.head      = nn.Conv2d(embed_dim, num_classes, kernel_size=1)

    def forward(self, patch_tokens):
        B, N, D = patch_tokens.shape
        g = self.grid_size
        x = patch_tokens.permute(0, 2, 1).reshape(B, D, g, g)
        x = self.head(x)
        return F.interpolate(
            x, size=(self.img_size, self.img_size),
            mode="bilinear", align_corners=False,
        )


def boundary_f1(pred_mask, gt_mask, num_classes, dilation=1):
    from scipy.ndimage import binary_dilation as bd
    bf1 = []
    for c in range(num_classes):
        p = (pred_mask == c).astype(np.uint8)
        g = (gt_mask   == c).astype(np.uint8)
        if g.sum() == 0:
            continue
        pb    = np.logical_xor(
            p, bd(p, iterations=dilation)).astype(np.uint8)
        gb    = np.logical_xor(
            g, bd(g, iterations=dilation)).astype(np.uint8)
        inter = (pb & gb).sum()
        denom = pb.sum() + gb.sum()
        if denom == 0:
            continue
        bf1.append(2 * inter / (denom + 1e-8))
    return float(np.mean(bf1)) if bf1 else 0.0


def eval_segmentation(backbone_path, eurosat_dir,
                      num_classes=10, epochs=30):
    device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    encoder = load_backbone(backbone_path, device)
    train_tf, val_tf = get_eval_transforms()

    split_dir = ensure_split(eurosat_dir, CFG["val_split"])
    train_ds  = datasets.ImageFolder(
        os.path.join(split_dir, "train"), transform=train_tf)
    val_ds    = datasets.ImageFolder(
        os.path.join(split_dir, "val"),   transform=val_tf)
    train_ldr = make_loader(train_ds, 64, shuffle=True, drop_last=True)
    val_ldr   = make_loader(val_ds,   64, shuffle=False)

    decoder = SegDecoder(
        CFG["embed_dim"], num_classes,
        patch_size = CFG["patch_size"],
        img_size   = CFG["img_size"],
    ).to(device)
    opt   = torch.optim.Adam(decoder.parameters(), lr=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    use_amp = AMP_DTYPE is not None

    def get_patch_tokens(x):
        with torch.inference_mode():
            latent, _, _ = encoder(
                x.to(device, non_blocking=True), mask_ratio=0.0)
            return latent[:, 1:]   # drop CLS → (B, 100, 384)

    for epoch in range(epochs):
        decoder.train()
        for x, y in train_ldr:
            tokens = get_patch_tokens(x)
            y      = y.to(device, non_blocking=True)
            with torch.amp.autocast(
                    "cuda", dtype=AMP_DTYPE, enabled=use_amp):
                logits = decoder(tokens)
                target = y.view(-1, 1, 1).expand(
                    -1, CFG["img_size"], CFG["img_size"])
                loss   = F.cross_entropy(logits, target)
            opt.zero_grad()
            loss.backward()
            opt.step()
        sched.step()
        if (epoch + 1) % 10 == 0:
            print(f"  Seg epoch {epoch+1}/{epochs}", flush=True)

    decoder.eval()
    confusion = np.zeros((num_classes, num_classes), dtype=np.int64)
    all_bf1   = []
    with torch.inference_mode():
        for x, y in val_ldr:
            tokens = get_patch_tokens(x)
            pred   = decoder(tokens).argmax(1).cpu().numpy()
            label  = y.numpy()
            for b in range(pred.shape[0]):
                gt_map = np.full_like(pred[b], label[b])
                for i in range(num_classes):
                    for j in range(num_classes):
                        confusion[i, j] += (
                            (gt_map == i) & (pred[b] == j)).sum()
                all_bf1.append(
                    boundary_f1(pred[b], gt_map, num_classes))

    iou_per_class = []
    for c in range(num_classes):
        tp    = confusion[c, c]
        fp    = confusion[:, c].sum() - tp
        fn    = confusion[c, :].sum() - tp
        denom = tp + fp + fn
        if denom > 0:
            iou_per_class.append(tp / denom)

    miou     = float(np.mean(iou_per_class)) * 100
    mean_bf1 = float(np.mean(all_bf1)) * 100
    print(
        f"  Segmentation  mIoU={miou:.2f}%  "
        f"Boundary-F1={mean_bf1:.2f}%"
    )
    return {"miou": round(miou, 2), "boundary_f1": round(mean_bf1, 2)}


# =============================================================================
# §4.3  RETRIEVAL PERFORMANCE
# =============================================================================

def haversine_km(lat1, lon1, lat2, lon2):
    R, d = 6371.0, math.radians
    dlat = d(lat2 - lat1)
    dlon = d(lon2 - lon1)
    a    = (math.sin(dlat / 2) ** 2 +
            math.cos(d(lat1)) * math.cos(d(lat2)) *
            math.sin(dlon / 2) ** 2)
    return R * 2 * math.asin(math.sqrt(a))


def mean_average_precision(sim_matrix, labels):
    N  = sim_matrix.shape[0]
    sm = sim_matrix.clone()
    sm.fill_diagonal_(-1e9)
    order   = sm.argsort(dim=1, descending=True)
    ap_list = []
    for i in range(N):
        gt    = (labels[order[i]] == labels[i])
        n_rel = gt.sum().item()
        if n_rel == 0:
            continue
        n_correct  = 0
        precisions = []
        for rank, hit in enumerate(gt.tolist(), 1):
            if hit:
                n_correct += 1
                precisions.append(n_correct / rank)
        ap_list.append(sum(precisions) / n_rel)
    return float(np.mean(ap_list)) * 100 if ap_list else 0.0


def eval_retrieval(backbone_path, eurosat_dir,
                   gps_csv=None, ks=(1, 5, 10),
                   geo_thresholds_km=(1, 5, 10)):
    device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    encoder = load_backbone(backbone_path, device)
    _, val_tf = get_eval_transforms()

    split_dir = ensure_split(eurosat_dir, CFG["val_split"])
    val_ds    = datasets.ImageFolder(
        os.path.join(split_dir, "val"), transform=val_tf)
    val_ldr   = make_loader(val_ds, 256, shuffle=False)

    print("  Extracting gallery features…")
    feats, labels = extract_features(encoder, val_ldr, device)
    feats = F.normalize(feats.float(), dim=-1)
    sim   = feats @ feats.T
    N     = feats.shape[0]

    results     = {}
    sim_no_diag = sim.clone()
    sim_no_diag.fill_diagonal_(-1e9)
    topk_max    = max(ks)
    top_indices = sim_no_diag.topk(topk_max, dim=1).indices

    for k in ks:
        top_k_labels = labels[top_indices[:, :k]]
        correct = (
            (top_k_labels == labels.unsqueeze(1)).any(dim=1).sum().item())
        recall  = 100.0 * correct / N
        results[f"recall@{k}"] = round(recall, 2)
        print(f"  Recall@{k} = {recall:.2f}%")

    mAP = mean_average_precision(sim, labels)
    results["mAP"] = round(mAP, 2)
    print(f"  mAP = {mAP:.2f}%")

    if gps_csv is not None:
        import pandas as pd
        gps_df       = pd.read_csv(gps_csv)
        img_paths    = [val_ds.samples[i][0] for i in range(N)]
        fname_to_gps = {
            row["filename"]: (row["lat"], row["lon"])
            for _, row in gps_df.iterrows()
        }
        errors_km = []
        geo_hits  = {k: {eps: 0 for eps in geo_thresholds_km} for k in ks}
        n_valid   = 0
        for i in range(N):
            q_fname = os.path.basename(img_paths[i])
            if q_fname not in fname_to_gps:
                continue
            n_valid += 1
            q_lat, q_lon = fname_to_gps[q_fname]
            top_fnames   = [
                os.path.basename(img_paths[j])
                for j in top_indices[i, :topk_max].tolist()
            ]
            if top_fnames[0] in fname_to_gps:
                r_lat, r_lon = fname_to_gps[top_fnames[0]]
                errors_km.append(
                    haversine_km(q_lat, q_lon, r_lat, r_lon))
            for k in ks:
                cands = [f for f in top_fnames[:k] if f in fname_to_gps]
                for eps in geo_thresholds_km:
                    if any(
                        haversine_km(q_lat, q_lon, *fname_to_gps[f]) <= eps
                        for f in cands
                    ):
                        geo_hits[k][eps] += 1
        if errors_km:
            med = float(np.median(errors_km)) * 1000
            p90 = float(np.percentile(errors_km, 90)) * 1000
            results["median_loc_error_m"] = round(med, 1)
            results["p90_loc_error_m"]    = round(p90, 1)
            print(
                f"  Median loc. error = {med:.1f} m  (P90 = {p90:.1f} m)")
        if n_valid > 0:
            for k in ks:
                for eps in geo_thresholds_km:
                    r = 100.0 * geo_hits[k][eps] / n_valid
                    results[f"geo_recall@{k}_{eps}km"] = round(r, 2)
                    print(
                        f"  Geo-Recall@{k} ({eps} km) = {r:.2f}%")

    return results


# =============================================================================
# §4.4  REPRESENTATION QUALITY
# =============================================================================

def effective_rank(feats):
    f       = feats - feats.mean(0)
    cov     = (f.T @ f) / (feats.shape[0] - 1)
    eigvals = torch.linalg.eigvalsh(cov.float()).clamp(min=0)
    eigvals = eigvals / eigvals.sum().clamp(min=1e-8)
    eigvals = eigvals[eigvals > 1e-9]
    entropy = -(eigvals * eigvals.log()).sum()
    return math.exp(entropy.item())


def uniformity_score(feats):
    feats = F.normalize(feats.float(), dim=-1)
    if feats.shape[0] > 2000:
        feats = feats[torch.randperm(feats.shape[0])[:2000]]
    sq = torch.cdist(feats, feats, p=2).pow(2)
    return round(sq.mul(-2).exp().mean().log().item(), 4)


def eval_representation_quality(backbone_path, eurosat_dir,
                                 knn_k=20, num_classes=10):
    device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    encoder = load_backbone(backbone_path, device)
    _, val_tf = get_eval_transforms()

    split_dir = ensure_split(eurosat_dir, CFG["val_split"])
    train_ds  = datasets.ImageFolder(
        os.path.join(split_dir, "train"), transform=val_tf)
    val_ds    = datasets.ImageFolder(
        os.path.join(split_dir, "val"),   transform=val_tf)
    train_ldr = make_loader(train_ds, 256, shuffle=False)
    val_ldr   = make_loader(val_ds,   256, shuffle=False)

    print("  Extracting features for representation quality…")
    train_feats, train_labels = extract_features(
        encoder, train_ldr, device)
    val_feats,   val_labels   = extract_features(
        encoder, val_ldr,   device)
    train_n = F.normalize(train_feats.float(), dim=-1)
    val_n   = F.normalize(val_feats.float(),   dim=-1)

    # P5: batched kNN via scatter_add
    sim       = val_n @ train_n.T
    knn_preds = batched_knn_predict(
        sim.to(device), train_labels, knn_k, num_classes)
    knn_acc   = 100.0 * (
        knn_preds.cpu() == val_labels).float().mean().item()
    print(f"  kNN accuracy (k={knn_k}) = {knn_acc:.2f}%")

    eff_rank = effective_rank(val_feats.float())
    print(f"  Effective rank = {eff_rank:.1f}")

    unif = uniformity_score(val_n)
    print(f"  Uniformity = {unif:.4f}")

    aug = _base_aug(
        CFG["img_size"], CFG["crop_scale"],
        CFG["jitter_strength"], CFG["blur_prob"],
    )
    two_view_ds  = TwoViewDataset(
        os.path.join(split_dir, "val"), aug)
    two_view_ldr = make_loader(two_view_ds, 256, shuffle=False)
    align_scores = []
    with torch.inference_mode():
        for x1, x2, _ in two_view_ldr:
            z1 = F.normalize(extract_feat(
                encoder, x1.to(device, non_blocking=True)), dim=-1)
            z2 = F.normalize(extract_feat(
                encoder, x2.to(device, non_blocking=True)), dim=-1)
            align_scores.append(
                (z1 - z2).pow(2).sum(dim=-1).mean().item())
    alignment = round(float(np.mean(align_scores)), 4)
    print(f"  Alignment = {alignment:.4f}")

    return {
        "knn_acc":        round(knn_acc, 2),
        "effective_rank": round(eff_rank, 1),
        "uniformity":     unif,
        "alignment":      alignment,
    }


# =============================================================================
# §4.5  BAND MISMATCH ROBUSTNESS
# =============================================================================

class BandAdapterBackbone(nn.Module):
    """Trainable 1×1 conv: N spectral bands → 3 RGB, then frozen encoder.
    Fine-tuned 5 epochs on MS data (P8, matches DINO §4.5)."""
    def __init__(self, encoder, in_channels=13, out_channels=3):
        super().__init__()
        self.adapter = nn.Conv2d(
            in_channels, out_channels, kernel_size=1, bias=False)
        self.encoder = encoder
        nn.init.kaiming_normal_(self.adapter.weight)

    def forward(self, x):
        return extract_feat(self.encoder, self.adapter(x))


def eval_band_mismatch(backbone_path, eurosat_rgb_dir,
                       eurosat_ms_dir=None, num_classes=10):
    device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    encoder = load_backbone(backbone_path, device)
    _, val_tf = get_eval_transforms()

    # RGB Recall@1
    split_dir = ensure_split(eurosat_rgb_dir, CFG["val_split"])
    val_ds    = datasets.ImageFolder(
        os.path.join(split_dir, "val"), transform=val_tf)
    val_ldr   = make_loader(val_ds, 256, shuffle=False)
    feats, labels = extract_features(encoder, val_ldr, device)
    feats = F.normalize(feats.float(), dim=-1)
    sim   = feats @ feats.T
    sim.fill_diagonal_(-1e9)
    recall_rgb = 100.0 * (
        labels[sim.argmax(dim=1)] == labels
    ).float().mean().item()
    print(f"  Recall@1 (RGB, 3-band) = {recall_rgb:.2f}%")

    recall_ms = None
    if eurosat_ms_dir and os.path.exists(eurosat_ms_dir):
        ms_ds = MultiSpectralDataset(
            eurosat_ms_dir, n_channels=13, img_size=CFG["img_size"])
        if len(ms_ds) == 0:
            print("  EuroSAT-MS: no files found — skipping.")
        else:
            adapter_bb = BandAdapterBackbone(
                encoder, in_channels=13, out_channels=3).to(device)
            ms_ldr_train = make_loader(ms_ds, 128, shuffle=True)

            tmp_head  = nn.Linear(CFG["embed_dim"], num_classes).to(device)
            opt_adapt = torch.optim.AdamW(
                list(adapter_bb.adapter.parameters()) +
                list(tmp_head.parameters()),
                lr=1e-3,
            )

            for _ in range(5):
                adapter_bb.adapter.train()
                tmp_head.train()
                for x, y in ms_ldr_train:
                    x = x.to(device, non_blocking=True)
                    y = y.to(device, non_blocking=True)
                    with torch.inference_mode():
                        feat = encoder(
                            adapter_bb.adapter(x))[0][:, 0]
                    opt_adapt.zero_grad()
                    F.cross_entropy(tmp_head(feat), y).backward()
                    opt_adapt.step()
            del tmp_head

            adapter_bb.eval()
            ms_ldr_eval      = make_loader(ms_ds, 128, shuffle=False)
            ms_feats, ms_lbl = [], []
            with torch.inference_mode():
                for x, y in ms_ldr_eval:
                    ms_feats.append(
                        adapter_bb(x.to(device, non_blocking=True))
                        .float().cpu())
                    ms_lbl.append(y)
            ms_feats  = F.normalize(torch.cat(ms_feats), dim=-1)
            ms_labels = torch.cat(ms_lbl)
            sim_ms    = ms_feats @ ms_feats.T
            sim_ms.fill_diagonal_(-1e9)
            recall_ms = 100.0 * (
                ms_labels[sim_ms.argmax(dim=1)] == ms_labels
            ).float().mean().item()
            print(f"  Recall@1 (MS, 13-band) = {recall_ms:.2f}%")
    else:
        print("  EuroSAT-MS directory not found — skipping.")

    delta = (round(recall_rgb - recall_ms, 2)
             if recall_ms is not None else None)
    if delta is not None:
        print(f"  Band mismatch penalty Δ = {delta:.2f}%")

    return {
        "recall1_rgb":         round(recall_rgb, 2),
        "recall1_ms":          round(recall_ms, 2) if recall_ms else None,
        "band_mismatch_delta": delta,
    }


# =============================================================================
# ── Full evaluation orchestrator ──────────────────────────────────────────────
# =============================================================================

def run_full_evaluation(backbone_path, eurosat_rgb_dir=None,
                        eurosat_ms_dir=None, gps_csv=None):
    eurosat_rgb_dir = eurosat_rgb_dir or CFG["eurosat_rgb_dir"]
    eurosat_ms_dir  = eurosat_ms_dir  or CFG.get("eurosat_ms_dir")
    all_results     = {
        "model":      "SatMAE-ViT-S",
        "checkpoint": backbone_path,
    }

    eval_steps = [
        (
            "§4.1  CLASSIFICATION (linear probe)",
            eval_classification,
            dict(
                backbone_path = backbone_path,
                eurosat_dir   = eurosat_rgb_dir,
                label_fracs   = (0.01, 0.1, 1.0),
                num_classes   = CFG["num_classes"],
            ),
        ),
        (
            "§4.2  SEGMENTATION",
            eval_segmentation,
            dict(
                backbone_path = backbone_path,
                eurosat_dir   = eurosat_rgb_dir,
                num_classes   = CFG["num_classes"],
            ),
        ),
        (
            "§4.3  RETRIEVAL PERFORMANCE",
            eval_retrieval,
            dict(
                backbone_path     = backbone_path,
                eurosat_dir       = eurosat_rgb_dir,
                gps_csv           = gps_csv,
                ks                = CFG["retrieval_ks"],
                geo_thresholds_km = CFG["geo_thresholds_km"],
            ),
        ),
        (
            "§4.4  REPRESENTATION QUALITY",
            eval_representation_quality,
            dict(
                backbone_path = backbone_path,
                eurosat_dir   = eurosat_rgb_dir,
                knn_k         = CFG["knn_k"],
                num_classes   = CFG["num_classes"],
            ),
        ),
        (
            "§4.5  BAND MISMATCH ROBUSTNESS",
            eval_band_mismatch,
            dict(
                backbone_path   = backbone_path,
                eurosat_rgb_dir = eurosat_rgb_dir,
                eurosat_ms_dir  = eurosat_ms_dir,
            ),
        ),
    ]

    for title, fn, kwargs in eval_steps:
        print("\n" + "=" * 60)
        print(title)
        print("=" * 60)
        # e.g. "§4.1" → "classification"
        section_key = (
            title.split("  ", 1)[1]          # drop "§4.x  "
                 .split(" ")[0]              # first word
                 .lower()
                 .strip("(")
        )
        all_results[section_key] = fn(**kwargs)

    out_path = os.path.join(CFG["output_dir"], "satmae_eval_results.json")
    with open(out_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nAll results saved to {out_path}")
    return all_results


# =============================================================================
# ── Entry point ───────────────────────────────────────────────────────────────
# =============================================================================

if __name__ == "__main__":
    train()

    BEST_CKPT = os.path.join(
        CFG["output_dir"], f"satmae_ep{CFG['epochs']}.pt")
    GPS_CSV = f"{DATA_DIR}/eurosat/eurosat_gps.csv"
    run_full_evaluation(
        BEST_CKPT,
        CFG["eurosat_rgb_dir"],
        CFG["eurosat_ms_dir"],
        GPS_CSV if os.path.exists(GPS_CSV) else None,
    )

/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()


Device: cuda  |  Workers: 4  |  AMP: torch.float16  |  compile: True  |  albu: True  |  batch: 512


/tmp/ipykernel_318/3669718463.py:572: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.blocks = nn.TransformerEncoder(


DataParallel on 2 GPUs.
torch.compile enabled on full SatMAE model.


W0417 10:27:40.549000 318 torch/_logging/_internal.py:1204] [0/0] Profiler function <class 'torch.autograd.profiler.record_function'> will be ignored
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/variables/functions.py:1946: UserWarning: Dynamo does not know how to trace the builtin `_thread.get_ident.` This function is either a Python builtin (e.g. _warnings.warn) or a third-party C/C++ Python extension (perhaps created with pybind).
If it is a Python builtin, please file an issue on GitHub so the PyTorch team can add support for it and see the next case for a workaround.
If it is a third-party C/C++ Python extension, please either wrap it into a PyTorch-understood custom operator (see https://pytorch.org/tutorials/advanced/custom_ops_landing_page.html for more details) or, if it is traceable, use `torch.compiler.allow_in_graph`.
  torch._dynamo.utils.warn_once(explanation + "\n" + "\n".join(hints))
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWa

Epoch [  1/200]  loss=1.0115  lr=6.00e-05  time=60s
Epoch [  2/200]  loss=0.8875  lr=1.20e-04  time=43s
Epoch [  3/200]  loss=0.8566  lr=1.80e-04  time=43s
Epoch [  4/200]  loss=0.8455  lr=2.40e-04  time=43s
Epoch [  5/200]  loss=0.8418  lr=3.00e-04  time=43s
Epoch [  6/200]  loss=0.8323  lr=3.60e-04  time=43s
Epoch [  7/200]  loss=0.8190  lr=4.20e-04  time=42s
Epoch [  8/200]  loss=0.8133  lr=4.80e-04  time=43s
Epoch [  9/200]  loss=0.8089  lr=5.40e-04  time=43s
Epoch [ 10/200]  loss=0.8425  lr=6.00e-04  time=42s
Epoch [ 11/200]  loss=0.8104  lr=6.60e-04  time=43s
Epoch [ 12/200]  loss=0.8073  lr=7.20e-04  time=43s
Epoch [ 13/200]  loss=0.8088  lr=7.80e-04  time=43s
Epoch [ 14/200]  loss=0.8116  lr=8.40e-04  time=43s
Epoch [ 15/200]  loss=0.7930  lr=9.00e-04  time=43s
Epoch [ 16/200]  loss=0.7958  lr=9.60e-04  time=43s
Epoch [ 17/200]  loss=0.9050  lr=1.02e-03  time=43s
Epoch [ 18/200]  loss=0.8777  lr=1.08e-03  time=43s
Epoch [ 19/200]  loss=0.8502  lr=1.14e-03  time=43s
Epoch [ 20/2

RuntimeError: Inference tensors cannot be saved for backward. Please do not use Tensors created in inference mode in computation tracked by autograd. To work around this, you can make a clone to get a normal tensor and use it in autograd, or use `torch.no_grad()` instead of `torch.inference_mode()`.

In [2]:
# ── Resume: re-run §4.5 with inference_mode bug fixed ────────────────────────
import json, os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets

def eval_band_mismatch(backbone_path, eurosat_rgb_dir,
                       eurosat_ms_dir=None, num_classes=10):
    device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    encoder   = load_backbone(backbone_path, device)
    _, val_tf = get_eval_transforms()

    # ── RGB Recall@1 ──────────────────────────────────────────────────────────
    split_dir     = ensure_split(eurosat_rgb_dir, CFG["val_split"])
    val_ds        = datasets.ImageFolder(os.path.join(split_dir, "val"), transform=val_tf)
    val_ldr       = make_loader(val_ds, 256, shuffle=False)
    feats, labels = extract_features(encoder, val_ldr, device)
    feats         = F.normalize(feats.float(), dim=-1)
    sim           = feats @ feats.T
    sim.fill_diagonal_(-1e9)
    recall_rgb = 100.0 * (labels[sim.argmax(dim=1)] == labels).float().mean().item()
    print(f"  Recall@1 (RGB, 3-band) = {recall_rgb:.2f}%")

    recall_ms = None
    if eurosat_ms_dir and os.path.exists(eurosat_ms_dir):
        ms_ds = MultiSpectralDataset(eurosat_ms_dir, n_channels=13, img_size=CFG["img_size"])
        if len(ms_ds) == 0:
            print("  EuroSAT-MS: no files found — skipping.")
        else:
            adapter_bb   = BandAdapterBackbone(encoder, in_channels=13, out_channels=3).to(device)
            ms_ldr_train = make_loader(ms_ds, 128, shuffle=True)
            tmp_head     = nn.Linear(CFG["embed_dim"], num_classes).to(device)
            opt_adapt    = torch.optim.AdamW(
                list(adapter_bb.adapter.parameters()) + list(tmp_head.parameters()), lr=1e-3)

            # FIX: inference_mode() makes tensors unusable for backward().
            # Run the adapter with grad; isolate the frozen encoder with no_grad()
            # and .detach() so gradients flow only through adapter & tmp_head.
            for _ in range(5):
                adapter_bb.adapter.train()
                tmp_head.train()
                for x, y in ms_ldr_train:
                    x       = x.to(device, non_blocking=True)
                    y       = y.to(device, non_blocking=True)
                    adapted = adapter_bb.adapter(x)          # grad tracked here
                    with torch.no_grad():
                        feat = encoder(adapted)[0][:, 0]     # encoder is frozen
                    feat = feat.detach().requires_grad_(True) # allow head grads
                    opt_adapt.zero_grad()
                    F.cross_entropy(tmp_head(feat), y).backward()
                    opt_adapt.step()
            del tmp_head

            # ── MS Recall@1 ───────────────────────────────────────────────────
            adapter_bb.eval()
            ms_ldr_eval      = make_loader(ms_ds, 128, shuffle=False)
            ms_feats, ms_lbl = [], []
            with torch.inference_mode():
                for x, y in ms_ldr_eval:
                    ms_feats.append(adapter_bb(x.to(device, non_blocking=True)).float().cpu())
                    ms_lbl.append(y)
            ms_feats  = F.normalize(torch.cat(ms_feats), dim=-1)
            ms_labels = torch.cat(ms_lbl)
            sim_ms    = ms_feats @ ms_feats.T
            sim_ms.fill_diagonal_(-1e9)
            recall_ms = 100.0 * (ms_labels[sim_ms.argmax(dim=1)] == ms_labels).float().mean().item()
            print(f"  Recall@1 (MS, 13-band) = {recall_ms:.2f}%")
    else:
        print("  EuroSAT-MS directory not found — skipping.")

    delta = round(recall_rgb - recall_ms, 2) if recall_ms is not None else None
    if delta is not None:
        print(f"  Band mismatch penalty Δ = {delta:.2f}%")

    return {
        "recall1_rgb":         round(recall_rgb, 2),
        "recall1_ms":          round(recall_ms, 2) if recall_ms else None,
        "band_mismatch_delta": delta,
    }

# ── Re-run §4.5 and merge into existing results ───────────────────────────────
print("=" * 60)
print("§4.5  BAND MISMATCH ROBUSTNESS  (resumed)")
print("=" * 60)

BEST_CKPT = os.path.join(CFG["output_dir"], f"satmae_ep{CFG['epochs']}.pt")

band_result = eval_band_mismatch(
    BEST_CKPT,
    CFG["eurosat_rgb_dir"],
    CFG.get("eurosat_ms_dir"),
)

# Merge into existing partial results (§4.1–4.4 already saved)
out_path = os.path.join(CFG["output_dir"], "satmae_eval_results.json")
if os.path.exists(out_path):
    with open(out_path) as f:
        all_results = json.load(f)
else:
    all_results = {"model": "SatMAE-ViT-S", "checkpoint": BEST_CKPT}

all_results["band"] = band_result

with open(out_path, "w") as f:
    json.dump(all_results, f, indent=2)
print(f"\nResults saved to {out_path}")

§4.5  BAND MISMATCH ROBUSTNESS  (resumed)
  Recall@1 (RGB, 3-band) = 74.07%
  Recall@1 (MS, 13-band) = 58.70%
  Band mismatch penalty Δ = 15.37%

Results saved to /kaggle/working/satmae/satmae_eval_results.json
